# Limpieza de Base de Datos de Talux
Equipo 6

## Parte 1

## Importar liberías y Excel

In [1]:
import pandas as pd
import numpy as np
import re
import logging

OUT_DIR = "Limpieza_Output"

logger = logging.getLogger()
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
    handler.close()

logging.basicConfig(
    filename = 'Loggs_parte1.txt',
    level = logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

logger = logging.getLogger()

### Cam, Cam2 y Cam 3

In [2]:
try:
    cam = pd.read_excel('TEC concentrado_base de datos.xlsx', sheet_name= 'Cam')
    cam2 = pd.read_excel('TEC concentrado_base de datos.xlsx', sheet_name= 'Cam2')
    cam3 = pd.read_excel('TEC concentrado_base de datos.xlsx', sheet_name= 'Cam3')
    logger.info('Cam, Cam2 y Cam3 importados exitosamente')
except Exception as e:
    logger.error(f'Error importing file: {e}')

### 2020, base y Leads

In [3]:
try:
    _2020 = pd.read_excel('TEC concentrado_base de datos.xlsx', sheet_name= '2020')
    base = pd.read_excel('TEC concentrado_base de datos.xlsx', sheet_name= 'base ', header = None,
                          names = ['VIN','Modelo','Año Vehiculo', 'Color','Nombre cliente','C.P.','Telefono 1','Telefono 2','E-mail'])
    leads = pd.read_excel('TEC concentrado_base de datos.xlsx', sheet_name='Leads')
    logger.info('2020, base y leads importados exitosamente')
except Exception as e:
    logger.error(f'Error importing file: {e}')

In [4]:
# Se renombran las columnas para que queden con los mismos nombres
cam.rename(
    columns={'Modelo V.N.': 'Modelo', 'Localidad.': 'Localidad', 'Teléfono': 'Telefono 1', 'Teléfono.1': 'Telefono 2'},
    inplace=True)
cam2.rename(columns={'Modelo V.N.': 'Modelo', 'Localidad.fis': 'Localidad', 'Teléfono': 'Telefono 1',
                     'Teléfono.1': 'Telefono 2'}, inplace=True)
cam3.rename(
    columns={'Modelo V.N.': 'Modelo', 'Teléfono': 'Telefono 1', 'Teléfono.1': 'Telefono 2', 'Año mod': 'Año Vehiculo'},
    inplace=True)
leads.rename(columns={'Nombre': 'Nombre cliente', 'Telefono': 'Telefono 1', 'Correo': 'E-mail', 'COMENTATIOS': 'Comentarios'},
             inplace=True)
_2020.rename(
    columns={'Año': 'Año Vehiculo', 'Teléfono': 'Telefono 1', 'Teléfono.1': 'Telefono 2',
             'Tot.cargo': 'T.factura'}, inplace=True)

cam_old = cam.copy()
cam2_old = cam2.copy()
cam3_old = cam3.copy()
_2020_old = _2020.copy()
base_old = base.copy()
leads_old = leads.copy()

## Eliminar duplicados

### Cam,Cam2,Cam3

In [5]:
try:
    cam.drop_duplicates(inplace = True)
    cam2.drop_duplicates(inplace = True)
    cam3.drop_duplicates(inplace = True)
    difRows = cam_old.shape[0] - cam.shape[0]
    logger.info(f'{difRows} duplicados de Cam eliminados exitosamente')
    difRows = cam2_old.shape[0] - cam2.shape[0]
    logger.info(f'{difRows} duplicados de Cam2 eliminados exitosamente')
    difRows = cam3_old.shape[0] - cam3.shape[0]
    logger.info(f'{difRows} duplicados de Cam3 eliminados exitosamente')
except Exception as e:
    logger.error(f'Error dropping duplicates: {e}')

cam_old = cam.copy()
cam2_old = cam2.copy()
cam3_old = cam3.copy()

### 2020, base, Leads

In [6]:
try:
    _2020.drop_duplicates(inplace = True)
    base.drop_duplicates(inplace = True)
    leads.drop_duplicates(inplace = True)
    logger.info('Duplicados de Cam, 2020, base y Leads eliminados exitosamente')

    difRows = _2020_old.shape[0] - _2020.shape[0]
    logger.info(f'{difRows} duplicados de 2020 eliminados exitosamente')
    difRows = base_old.shape[0] - base.shape[0]
    logger.info(f'{difRows} duplicados de Base eliminados exitosamente')
    difRows = leads_old.shape[0] - leads.shape[0]
    logger.info(f'{difRows} duplicados de Leads eliminados exitosamente')

except Exception as e:
    logger.error(f'Error dropping duplicates: {e}')

_2020_old = _2020.copy()
base_old = base.copy()
leads_old = leads.copy()

## Eliminar espacios con formatos inusuales

In [7]:
def replaceRareSpaces(df, columnas):
    for col in columnas:
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

### cam1, cam2, cam3

In [8]:
try:
    replaceRareSpaces(cam, ['Modelo', 'Color', 'Localidad'])
    replaceRareSpaces(cam2, ['Modelo', 'Color', 'Colonia', 'Localidad'])
    replaceRareSpaces(cam3, ['Modelo', 'Color'])
    logger.info('Espacios con formatos inusuales estandarizados en Cam, Cam2 y Cam3 exitosamente')
except Exception as e:
    logger.error(f'Error transforming unusual spaces: {e}')

cam_old = cam.copy()
cam2_old = cam2.copy()
cam3_old = cam3.copy()

### 2020, base, leads

In [9]:
try:
     replaceRareSpaces(_2020, ['Modelo'])
     replaceRareSpaces(base, ['Modelo', 'Color'])
     replaceRareSpaces(leads, ['Modelo','Comentarios'])
     logger.info('Espacios con formatos inusuales estandarizados en Cam, Cam2 y Cam3 exitosamente')
except Exception as e:
     logger.error(f'Error transforming inusual spaces: {e}')

_2020_old = _2020.copy()
base_old = base.copy()
leads_old = leads.copy()

## Aplicar Formatos

In [10]:
try:
    # VIN debe tener una longitud de 17 caracteres
    cam['VIN'] = cam['VIN'].where(cam['VIN'].str.len() == 17)
    cam2['VIN'] = cam2['VIN'].where(cam['VIN'].str.len() == 17)
    cam3['VIN'] = cam3['VIN'].where(cam3['VIN'].str.len() == 17)
    _2020['VIN'] = _2020['VIN'].where(_2020['VIN'].str.len() == 17)
    base['VIN'] = base['VIN'].where(base['VIN'].str.len() == 17)

    for nombre, i in {'Cam':[cam,cam_old], 'Cam2':[cam2,cam2_old], 'Cam3':[cam3,cam3_old], '2020':[_2020,_2020_old], 'Base':[base,base_old]}.items():
        nfixed = i[0]['VIN'].isnull().sum() - i[1]['VIN'].isnull().sum()
        logger.info(f'{nfixed} valores de VIN fueron transformados en nulos en {nombre}')


    # Año Vehiculo debe ser un int de 4 dígitos mayor que 1970 y menor o igual que 2027.
    cam3['Año Vehiculo'] = cam3['Año Vehiculo'].where((cam3['Año Vehiculo'] > 1970) & (cam3['Año Vehiculo'] <= 2027))
    cam3['Año Vehiculo'] =  cam3['Año Vehiculo'].astype('Int64')
    _2020['Año Vehiculo'] = _2020['Año Vehiculo'].where((_2020['Año Vehiculo'] > 1970) & (_2020['Año Vehiculo'] <= 2026))
    _2020['Año Vehiculo'] = _2020['Año Vehiculo'].astype('Int64')
    base['Año Vehiculo'] = base['Año Vehiculo'].where((base['Año Vehiculo'] > 1970) & (base['Año Vehiculo'] <= 2026))
    base['Año Vehiculo'] = base['Año Vehiculo'].astype('Int64')

    for nombre, i in {'Cam3':[cam3,cam3_old], '2020':[_2020,_2020_old], 'Base':[base,base_old]}.items():
        nfixed = i[0]['Año Vehiculo'].isnull().sum() - i[1]['Año Vehiculo'].isnull().sum()
        logger.info(f'{nfixed} valores de Año Vehiculo fueron transformados en nulos en {nombre}')


    # Cliente debe ser un int
    cam['Cliente'] =  cam['Cliente'].astype('Int64')

    for nombre, i in {'Cam':[cam,cam_old]}.items():
        nfixed = i[0]['Cliente'].isnull().sum() - i[1]['Cliente'].isnull().sum()
        logger.info(f'{nfixed} valores de Cliente fueron transformados en nulos en {nombre}')


    # Nombre cliente debe llevar únicamente letras (en ese caso no aplica porque los nombres están censurados)
    #_2020['Nombre cliente'] = _2020['Nombre cliente'].where(_2020['Nombre cliente'].str.match(r'[a-zA-ZáÁéÉíÍóÓúÚñÑ]+'))
    #base['Nombre cliente'] = base['Nombre cliente'].where(base['Nombre cliente'].str.match(r'[a-zA-ZáÁéÉíÍóÓúÚñÑ]+'))
    #leads['Nombre cliente'] = leads['Nombre cliente'].where(leads['Nombre cliente'].str.match(r'[a-zA-ZáÁéÉíÍóÓúÚñÑ]+'))


    # C.P. debe tener 5 cifras y ser str
    cam['C.P.'] =  cam['C.P.'].astype("Int64").astype(str)
    cam['C.P.'] = cam['C.P.'].where(cam['C.P.'].str.len() == 5)
    cam2['C.P.'] =  cam2['C.P.'].astype("Int64").astype(str)
    cam2['C.P.'] = cam2['C.P.'].where(cam2['C.P.'].str.len() == 5)
    cam3['C.P.'] =  cam3['C.P.'].astype("Int64").astype(str)
    cam3['C.P.'] = cam3['C.P.'].where(cam3['C.P.'].str.len() == 5)
    base['C.P.'] =  base['C.P.'].astype(str)
    base['C.P.'] = base['C.P.'].where(base['C.P.'].str.len() == 5)

    for nombre, i in {'Cam':[cam,cam_old], 'Cam2':[cam2,cam2_old], 'Cam3':[cam3,cam3_old], 'Base':[base,base_old]}.items():
        nfixed = i[0]['C.P.'].isnull().sum() - i[1]['C.P.'].isnull().sum()
        logger.info(f'{nfixed} valores de C.P. fueron transformados en nulos en {nombre}')


    # Colonia debe tener al menos 3 letras
    cam2['Colonia'] = cam2['Colonia'].where(cam2['Colonia'].str.match(r'[a-zA-Z]{2,}'))

    nfixed = cam2['Colonia'].isnull().sum() - cam2['Colonia'].isnull().sum()
    logger.info(f'{nfixed} valores de Colonia fueron transformados en nulos en cam2')


    # Localidad debe tener nombres estandarizados, sobre todo con la CDMX.
    cdmx_nombres = ['MEXICO','CIUDAD DE','DISTRITO F']
    cam['Localidad'] = cam['Localidad'].str.replace({name: 'CDMX' for name in cdmx_nombres})
    cam['Localidad'] = cam['Localidad'].str.replace('ESTADO DE','EdoMex', regex=False)
    cam['Localidad'] = cam['Localidad'].str.replace('MONTERREY','NUEVO LEON', regex=False)

    cdmx_nombres = ['DISTRITO FEDERAL','CIUDAD DE MEXICO','MEXICO','CIUDAD DE MEXICO.']
    cam2['Localidad'] = cam2['Localidad'].str.replace({name: 'CDMX' for name in cdmx_nombres})
    cam2['Localidad'] = cam2['Localidad'].str.replace('MONTERREY','NUEVO LEON', regex=False)


    # Telefono debe ser de 10 dígitos, sin espacios y sin guiones.
    cam['Telefono 1'] = cam['Telefono 1'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    cam['Telefono 1'] = cam['Telefono 1'].where(cam['Telefono 1'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    cam2['Telefono 1'] = cam2['Telefono 1'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    cam2['Telefono 1'] = cam2['Telefono 1'].where(cam2['Telefono 1'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    cam3['Telefono 1'] = cam3['Telefono 1'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    cam3['Telefono 1'] = cam3['Telefono 1'].where(cam3['Telefono 1'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    _2020['Telefono 1'] = _2020['Telefono 1'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    _2020['Telefono 1'] = _2020['Telefono 1'].where(_2020['Telefono 1'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    base['Telefono 1'] = base['Telefono 1'].astype(str)
    base['Telefono 1'] = base['Telefono 1'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    base['Telefono 1'] = base['Telefono 1'].where(base['Telefono 1'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    leads['Telefono 1'] = leads['Telefono 1'].astype('Int64').astype(str)
    leads['Telefono 1'] = leads['Telefono 1'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    leads['Telefono 1'] = leads['Telefono 1'].where(leads['Telefono 1'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    for nombre, i in {'Cam':[cam,cam_old], 'Cam2':[cam2,cam2_old], 'Cam3':[cam3,cam3_old], '2020':[_2020,_2020_old], 'Base':[base,base_old], 'Leads':[leads,leads_old]}.items():
        nfixed = i[0]['Telefono 1'].isnull().sum() - i[1]['Telefono 1'].isnull().sum()
        logger.info(f'{nfixed} valores de Telefono 1 fueron transformados en nulos en {nombre}')


    # Telefono 2
    cam['Telefono 2'] = cam['Telefono 2'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    cam['Telefono 2'] = cam['Telefono 2'].where(cam['Telefono 2'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    cam2['Telefono 2'] = cam2['Telefono 2'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    cam2['Telefono 2'] = cam2['Telefono 2'].where(cam2['Telefono 2'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    cam3['Telefono 2'] = cam3['Telefono 2'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    cam3['Telefono 2'] = cam3['Telefono 2'].where(cam3['Telefono 2'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    _2020['Telefono 2'] = _2020['Telefono 2'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    _2020['Telefono 2'] = _2020['Telefono 2'].where(_2020['Telefono 2'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    base['Telefono 2'] = base['Telefono 2'].astype('Int64').astype(str)
    base['Telefono 2'] = base['Telefono 2'].str.replace(r'\D', '', regex=True) #Quitar sólo lo que no sea num.
    base['Telefono 2'] = base['Telefono 2'].where(base['Telefono 2'].str.len() == 10) #Dejar solo aquellos de 10 dígitos

    for nombre, i in {'Cam':[cam,cam_old], 'Cam2':[cam2,cam2_old], 'Cam3':[cam3,cam3_old], '2020':[_2020,_2020_old], 'Base':[base,base_old]}.items():
        nfixed = i[0]['Telefono 2'].isnull().sum() - i[1]['Telefono 2'].isnull().sum()
        logger.info(f'{nfixed} valores de Telefono 2 fueron transformados en nulos en {nombre}')


    # Email debe de tener la estructura de uno
    cam['E-mail'] = cam['E-mail'].where(cam['E-mail'].str.match(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-]+\.[a-zA-Z]+'))
    cam2['E-mail'] = cam2['E-mail'].where(cam2['E-mail'].str.match(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-]+\.[a-zA-Z]+'))
    cam3['E-mail'] = cam3['E-mail'].where(cam3['E-mail'].str.match(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-]+\.[a-zA-Z]+'))
    _2020['E-mail'] = _2020['E-mail'].where(_2020['E-mail'].str.match(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-]+\.[a-zA-Z]+'))
    base['E-mail'] = base['E-mail'].where(base['E-mail'].str.match(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-]+\.[a-zA-Z]+'))
    leads['E-mail'] = leads['E-mail'].where(leads['E-mail'].str.match(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9._%+-]+\.[a-zA-Z]+'))

    for nombre, i in {'Cam':[cam,cam_old], 'Cam2':[cam2,cam2_old], 'Cam3':[cam3,cam3_old], '2020':[_2020,_2020_old], 'Base':[base,base_old], 'Leads':[leads,leads_old]}.items():
        nfixed = i[0]['E-mail'].isnull().sum() - i[1]['E-mail'].isnull().sum()
        logger.info(f'{nfixed} valores de E-mail fueron transformados en nulos en {nombre}')


    # Convertir los números negativos a positivos en T.Factura
    cam['T.factura'] = cam['T.factura'].astype(str).str.replace('-', '', regex=True)
    cam['T.factura'] = cam['T.factura'].astype('float64')

    cam2['T.factura'] = cam2['T.factura'].astype(str).str.replace('-', '', regex=True)
    cam2['T.factura'] = cam2['T.factura'].astype('float64')

    _2020['T.factura'] = _2020['T.factura'].astype(str).str.replace('-', '', regex=True)
    _2020['T.factura'] = _2020['T.factura'].astype('float64')

    for nombre, i in {'Cam':[cam,cam_old], 'Cam2':[cam2,cam2_old], '2020':[_2020,_2020_old]}.items():
        nfixed = i[0]['T.factura'].isnull().sum() - i[1]['T.factura'].isnull().sum()
        logger.info(f'{nfixed} valores de T.factura fueron transformados en nulos en {nombre}')

except Exception as e:
    logger.error(f'Error formating variables: {e}')

## Parte 2

In [11]:
from datetime import datetime

# =========================
# CONFIG
# =========================
INPUT_PATH = "TEC concentrado_base de datos.xlsx"   # ruta en aleman

TARGETS_F = ["f01", "f02", "f03"]
TARGETS_YEARS = ["2025", "2024", "2023"]

#os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================
def norm_col(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[\s\-/]+", "_", s)
    s = re.sub(r"[^a-z0-9_áéíóúñü]+", "", s)
    return s

def find_col(cols, candidates):
    ncols = {c: norm_col(c) for c in cols}
    for cand in candidates:
        candn = norm_col(cand)
        for c, cn in ncols.items():
            if cn == candn:
                return c
        for c, cn in ncols.items():
            if candn in cn:
                return c
    return None

def strip_df(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.copy()
    for col in df2.columns:
        df2[col] = df2[col].map(lambda x: x.strip() if isinstance(x, str) else x)
    return df2

def clean_phone(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return np.nan

    plus = s.startswith("+")
    digits = re.sub(r"\D+", "", s)
    if digits == "":
        return np.nan

    if len(digits) == 10:
        return digits
    elif digits.startswith("+52") and len(digits) == 13:
        digits2 = digits[2:]
        return digits2
    elif digits.startswith("52") and len(digits) == 12:
        digits2 = digits[3:]
        return digits2
    else:
        return np.nan

EMAIL_RE = re.compile(r"^[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}$", re.I)

def clean_email(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "" or s in {"nan", "none", "null"}:
        return np.nan

    s = s.strip(" ,;.")
    s = s.replace(" ", "")

    if EMAIL_RE.match(s):
        return s

    parts = re.split(r"[;,/]+", s)
    for p in parts:
        p = p.strip().replace(" ", "")
        if EMAIL_RE.match(p):
            return p
    return np.nan

def write_log(path, lines):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

def promote_header_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    cols = list(df.columns)
    unnamed_ratio = sum(str(c).startswith("Unnamed") for c in cols) / max(1, len(cols))
    if unnamed_ratio < 0.8:
        return df

    for i in range(min(10, len(df))):
        row = df.iloc[i].tolist()
        nonnull = [x for x in row if isinstance(x, str) and x.strip() != ""]
        if len(nonnull) >= 2 and any(re.search(r"tel", str(x), re.I) for x in nonnull):
            new_cols = [str(x).strip() if isinstance(x, str) else f"col_{j}" for j, x in enumerate(row)]
            df2 = df.iloc[i + 1 :].copy()
            df2.columns = new_cols
            return df2

    return df

# =========================
# CLEANERS
# =========================
def clean_f_sheet(df: pd.DataFrame, sheet_name: str):
    log = [f"=== Sheet {sheet_name} ==="]

    df0 = df.copy()
    df0 = df0.replace(r"^\s*$", np.nan, regex=True)

    before_rows = len(df0)
    before_cols = df0.shape[1]

    df0 = df0.dropna(how="all")
    log.append(f"Rows: {before_rows} -> {len(df0)} after dropping fully empty rows.")
    df0 = strip_df(df0)

    vin_col = find_col(df0.columns, ["vin", "vehicle_identification_number", "num_vin"])
    modelo_col = find_col(df0.columns, ["modelo", "model"])

    if vin_col:
        df0[vin_col] = df0[vin_col].map(lambda x: re.sub(r"\s+", "", x.upper()) if isinstance(x, str) else x)

        invalid = df0[vin_col].map(lambda x: isinstance(x, str) and len(x) not in (0, 17))
        inv_count = int(invalid.sum())
        if inv_count:
            df0.loc[invalid, vin_col] = np.nan
        log.append(f"VIN column: {vin_col}. Invalid VIN length set to NaN: {inv_count}.")

        dup_mask = df0[vin_col].notna() & df0.duplicated(subset=[vin_col], keep="first")
        dup_count = int(dup_mask.sum())
        if dup_count:
            df0 = df0.loc[~dup_mask].copy()
        log.append(f"Duplicate VIN rows removed: {dup_count}.")
    else:
        log.append("VIN column: NOT FOUND (no VIN-based dedup applied).")

    if modelo_col:
        df0[modelo_col] = df0[modelo_col].map(lambda x: x.title() if isinstance(x, str) else x)
        log.append(f"Modelo column standardized to Title Case: {modelo_col}.")

    tel_col = find_col(df0.columns, ["telefono", "teléfono", "phone", "celular", "movil", "móvil"])
    mail_col = find_col(df0.columns, ["correo", "e-mail", "email", "mail"])

    if tel_col:
        old = df0[tel_col].copy()
        df0[tel_col] = df0[tel_col].map(clean_phone)
        changed = int((old.fillna("") != df0[tel_col].fillna("")).sum())
        log.append(f"Phone standardized in column {tel_col}. Cells changed: {changed}.")

    if mail_col:
        old = df0[mail_col].copy()
        df0[mail_col] = df0[mail_col].map(clean_email)
        changed = int((old.fillna("") != df0[mail_col].fillna("")).sum())
        log.append(f"Email standardized in column {mail_col}. Cells changed: {changed}.")

    # Regla F: si NO hay VIN, mail y telefono -> bye
    vin_missing = pd.Series([True] * len(df0), index=df0.index)
    tel_missing = pd.Series([True] * len(df0), index=df0.index)
    mail_missing = pd.Series([True] * len(df0), index=df0.index)

    if vin_col:
        vin_missing = df0[vin_col].isna()
    if tel_col:
        tel_missing = df0[tel_col].isna()
    if mail_col:
        mail_missing = df0[mail_col].isna()

    drop_mask = vin_missing & tel_missing & mail_missing
    drop_count = int(drop_mask.sum())
    df0 = df0.loc[~drop_mask].copy()
    log.append(f"Rows dropped (missing VIN+Tel+Email all at once): {drop_count}.")

    log.append(f"Final shape: {df0.shape[0]} rows x {df0.shape[1]} cols (started with {before_cols} cols).")
    return df0, log

def clean_year_sheet_keep4(df: pd.DataFrame, sheet_name: str):
    """
    Años: conserva 4 columnas: telefono, correo, estatus, auto.
    Tel/correo se usan para filtrar (si ambos vacíos -> bye) y estandarizar.
    + dedup exacto por las 4 columnas.
    """
    log = [f"=== Sheet {sheet_name} ==="]

    df0 = df.copy()
    df0 = df0.replace(r"^\s*$", np.nan, regex=True)

    before_rows = len(df0)
    df0 = df0.dropna(how="all")
    log.append(f"Rows: {before_rows} -> {len(df0)} after dropping fully empty rows.")
    df0 = strip_df(df0)

    # Regla: >3 vacías bye (sobre columnas originales)
    empties = df0.isna().sum(axis=1)
    drop_mask = empties > 3
    drop_count = int(drop_mask.sum())
    df0 = df0.loc[~drop_mask].copy()
    log.append(f"Rows dropped for >3 empty cells (original columns): {drop_count}.")

    tel_col = find_col(df0.columns, ["teléfono", "telefono", "phone", "celular", "movil", "móvil"])
    mail_col = find_col(df0.columns, ["correo", "e-mail", "email", "mail"])
    est_col = find_col(df0.columns, ["estatus", "status"])
    auto_col = find_col(df0.columns, ["auto", "vehiculo", "vehículo", "car"])

    out = pd.DataFrame()
    out["telefono"] = df0[tel_col] if tel_col else np.nan
    out["correo"] = df0[mail_col] if mail_col else np.nan
    out["estatus"] = df0[est_col] if est_col else np.nan
    out["auto"] = df0[auto_col] if auto_col else np.nan

    out["telefono"] = out["telefono"].map(clean_phone)
    out["correo"] = out["correo"].map(clean_email)
    out["estatus"] = out["estatus"].map(lambda x: x.strip().title() if isinstance(x,str) else x)
    out["auto"] = out["auto"].map(lambda x: x.strip().upper() if isinstance(x,str) else x)

    # Regla años: si telefono y correo están vacíos -> bye
    both_empty = out["telefono"].isna() & out["correo"].isna()
    both_count = int(both_empty.sum())
    out = out.loc[~both_empty].copy()
    log.append(f"Rows dropped for telefono+correo both empty: {both_count}.")

    # No pierde variaciones de estatus/auto
    before = len(out)
    out = out.drop_duplicates(subset=["telefono", "correo", "estatus", "auto"], keep="first")
    log.append(f"Exact-duplicate rows removed (telefono+correo+estatus+auto): {before - len(out)}.")

    log.append(f"Kept columns: telefono, correo, estatus, auto (source: {tel_col}, {mail_col}, {est_col}, {auto_col}).")
    log.append(f"Final shape: {out.shape[0]} rows x {out.shape[1]} cols.")
    return out, log

# =========================
# MAIN
# =========================
run_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

logs_header = [
        f"Limpieza de datos de Talux parte 2: {run_ts}",
        "",
    ]

sheets = pd.read_excel(INPUT_PATH, sheet_name=None, dtype=str)

missing = [s for s in (TARGETS_F + TARGETS_YEARS) if s not in sheets]
if missing:
    raise ValueError(f"Missing sheets in workbook: {missing}. Available: {list(sheets.keys())}")

cleaned = {}
all_logs = []

    # f sheets
for s in TARGETS_F:
    df_clean, log = clean_f_sheet(sheets[s], s)
    cleaned[s] = df_clean
    all_logs += log + [""]

    # year sheets
for s in TARGETS_YEARS:
    df_prom = promote_header_if_needed(sheets[s])
    df_clean, log = clean_year_sheet_keep4(df_prom, s)
    cleaned[s] = df_clean
    all_logs += log + [""]

    # =========================
    # VARIABLES EN MEMORIA
    # =========================
f01_clean = cleaned["f01"]
f02_clean = cleaned["f02"]
f03_clean = cleaned["f03"]

y2023_clean = cleaned["2023"]
y2024_clean = cleaned["2024"]
y2025_clean = cleaned["2025"]

    # ONE MASTER LOG
master_log_path = "Loggs_Parte2.txt"
write_log(master_log_path, logs_header + all_logs)

print(" DONE. Outputs saved in:", OUT_DIR)
print("Master log:", master_log_path)

 DONE. Outputs saved in: Limpieza_Output
Master log: Loggs_Parte2.txt


In [12]:
f01 = cleaned['f01']
f02 = cleaned['f02']
f03 = cleaned['f03']
_2023 = cleaned['2023']
_2024 = cleaned['2024']
_2025 = cleaned['2025']

f01.rename(
    columns={'Modelo V.N.': 'Modelo', 'Localidad.': 'Localidad', 'Teléfono': 'Telefono 1', 'Teléfono.1': 'Telefono 2'},
    inplace=True)
f02.rename(
    columns={'Modelo V.N.': 'Modelo', 'Localidad.': 'Localidad', 'Teléfono': 'Telefono 1', 'Teléfono.1': 'Telefono 2'},
    inplace=True)
f03.rename(
    columns={'Modelo V.N.': 'Modelo', 'Localidad.': 'Localidad', 'Teléfono': 'Telefono 1', 'Teléfono.1': 'Telefono 2'},
    inplace=True)
_2023.rename(
    columns={'auto': 'Modelo', 'telefono': 'Telefono 1', 'correo': 'E-mail', 'estatus':'Estatus'},
    inplace=True)
_2024.rename(
    columns={'auto': 'Modelo', 'telefono': 'Telefono 1', 'correo': 'E-mail', 'estatus':'Estatus'},
    inplace=True)
_2025.rename(
    columns={'auto': 'Modelo', 'telefono': 'Telefono 1', 'correo': 'E-mail', 'estatus':'Estatus'},
    inplace=True)

In [13]:
# Eliminar espacios inusuales
try:
    replaceRareSpaces(f01, ['Modelo', 'Color', 'Colonia', 'Localidad'])
    replaceRareSpaces(f02, ['Modelo', 'Color', 'Colonia', 'Localidad'])
    replaceRareSpaces(f03, ['Modelo', 'Color', 'Colonia', 'Localidad'])
    logger.info('Espacios con formatos inusuales estandarizados en f01, f02 y f03 exitosamente')
except Exception as e:
    logger.error(f'Error transforming unusual spaces: {e}')

## Concatenar y Exportar CSV

In [20]:
try:
    dff = pd.concat([cam,cam2,cam3,_2020,base,leads,f01,f02,f03,_2023,_2024,_2025], ignore_index = True)
    dff.dropna(inplace=True, how='all')
    dff['Apellido'] = np.nan
    dff['Genero'] = np.random.choice(['M', 'F'], size=len(dff))
    dff['Fecha de nacimiento'] = np.nan
    dff['Marca Auto'] = np.nan
    dff['Empresa'] = np.nan
    dff['Giro empresa'] = np.nan
    dff['new_column'] = dff['other_column'].isna()

    dff.to_csv('ConglomeradoTalux.csv')
    logger.info('csv creado exitosamente')
except Exception as e:
    logger.error(f'Error exporting CSV: {e}')